# AutoRepro-Min — benchmark on Colab / Kaggle

Reduces a real Python project to a minimal reproduction of one failing
test, then scores it with the Gistify metric (Execution Fidelity).

**Read this before spending a session on it.** The benchmark is CPU-bound
in `pytest` subprocesses. Colab's free tier gives 2 vCPU and Kaggle 4,
against 10 on the laptop these numbers were measured on — so this will
most likely run *slower* here, not faster. It is worth doing to leave
your own machine free, to reproduce the numbers somewhere neutral, or to
run several configurations side by side in separate sessions.

Rough costs on a 10-core laptop: ~4 min per `requests` task, ~21 min per
`flask` task, ~1 h for all six. Budget more here.

On Kaggle, turn **Internet on** in the session settings — the benchmark
clones `psf/requests` and `pallets/flask`.

## 1. Get the code

`BRANCH` picks what you are testing. `master` is the known-good
sequential version; `parallel-oracle` adds the `--jobs` flag.

In [ ]:
BRANCH = 'parallel-oracle'   # or 'master'

!git clone --quiet --branch $BRANCH https://github.com/i-shantt/autorepro-min.git
%cd autorepro-min
!git log --oneline -3

## 2. What are we running on?

Worth knowing before reading any timing: every worker is a `pytest`
process, so the core count is the ceiling on `--jobs`.

In [ ]:
!python evaluation/cloud_bench.py --summary  # no results yet; prints the host check
import os; print('logical CPUs:', os.cpu_count())

## 3. Run it

One task per process, each result written separately — so a disconnect
does not cost you the whole run, and re-running this cell continues
from where it stopped rather than starting over.

Start with `--only requests` (four tasks, the fast repo). Drop the flag
for all six once you know the session survives.

In [ ]:
!python evaluation/cloud_bench.py --only requests

## 4. All six, with the parallel oracle

`--jobs N` validates Phase 4b candidates across N verified copies of the
project. The result is *identical* to `--jobs 1` — only runtime changes —
and it falls back to sequential if a copy cannot be verified to import
its own code. Omit `--jobs` to let it pick from the core count.

Only on the `parallel-oracle` branch.

In [ ]:
!python evaluation/cloud_bench.py --jobs 4

## 5. Results

Compare **queries and line counts**, not wall clock: those are
deterministic, wall clock is not comparable across machines.

Reference (M4 MacBook Air, sequential): 88.52% aggregate, 97.45% over the
code eligible for removal, 6/6 fidelity, 10,620 queries. The paper's best
reported execution fidelity is 58.7%.

In [ ]:
!python evaluation/cloud_bench.py --summary

## 6. Reduce your own project

Point it at a directory and the command that reproduces your bug. What
survives is a small tree that still reproduces it.

The command names the test, and that test plus any `conftest.py` above it
are protected — they are the statement of what must stay true, not code
to be reduced.

In [ ]:
# !python autorepro_min.py reduce-project /path/to/project \
#     --command 'python -m pytest tests/test_thing.py::test_case -x -q'